# DesInventar Analysis Notebook

This notebook processes DesInventar raw country data and produces harmonized indicators and hazard-level summaries for comparative analysis.

This notebook is developed with support of Copilot.

**Purpose**: Build a cleaned, grouped, and analysis-ready dataset from raw DesInventar exports.

**Input**:
- Zipped country dataset containing xlm file downloaded from DesInventar`raw/DI_export_*/`
- Filter of event types that are relevant for this project `unique_event_types_reviewed.csv`

**Output**:
- Records for all event types in all countries of intersted `extracted/desinventar_combined_translated_grouped.csv`
- Supporting plots in `plots/` for demonstration

## Setup

Import required libraries, define paths, and create helper plotting utilities used throughout the notebook.

In [ ]:
import os
import zipfile
import pandas as pd
import deepl
import time
import matplotlib.pyplot as plt
import numpy as np
import math

In [ ]:

# Path config
folder_path = r"raw/"
output_csv_folder = r"extracted/"
os.makedirs(output_csv_folder, exist_ok=True)
plots_output_dir = 'plots'
os.makedirs(plots_output_dir, exist_ok=True)


# XLM tag name to extract data from
all_dfs = []
zips_without_xml = []
tag_name = "fichas/TR"  # "fichas/TR", "lev0/TR", "lev2/TR", "lev3/TR"

In [ ]:
# Function to fix text encoding issues
def fix_text(s):
    if isinstance(s, str):
        try:
            return s.encode('latin1').decode('utf-8')
        except:
            return s
    return s


# Function to plot hazard data
def plot_hazard(
        hazard_name,
        df_melted,
        ncols,
        figsize_per,
        orientation,
        normalize,
        min_metric_value,
        save):
    df_h = df_melted[df_melted['event_type_grouped'] == hazard_name].copy()

    # pivot to country x metric
    pivot = df_h.pivot_table(index='country_code', columns='metric', values='value', aggfunc='first').fillna(0)
    metrics = [m for m in pivot.columns.tolist() if pivot[m].abs().max() > min_metric_value]
    if len(metrics) == 0:
        print(f'No metrics above threshold for hazard: {hazard_name}')
        return
    pivot = pivot[metrics]

    # normalization options
    if normalize == 'metric':
        for m in metrics:
            mx = pivot[m].abs().max()
            if mx > 0:
                pivot[m] = pivot[m] / mx
    elif normalize == 'country':
        pivot = pivot.div(pivot.max(axis=1).replace(0, 1), axis=0)

    countries = pivot.index.tolist()
    n_countries = len(countries)
    ncols = max(1, ncols)
    nrows = math.ceil(n_countries / ncols)

    fig_w = ncols * figsize_per[0]
    fig_h = nrows * figsize_per[1]
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), squeeze=False, constrained_layout=True)
    axes_flat = axes.flatten()

    for i, country in enumerate(countries):
        ax = axes_flat[i]
        values = pivot.loc[country, metrics].values
        nonzero_mask = (np.abs(values) > 0)  # mask of metrics that have non-zero values for this country"
        if orientation == 'horizontal':
            ax.barh(metrics, values, color='C0')
            # highlight y tick labels where value != 0"
            yticks = ax.get_yticklabels()
            for lbl, nz in zip(yticks, nonzero_mask):
                lbl.set_color('black' if nz else 'lightgray')
                # if nz:
                #     lbl.set_fontweight('bold')
        else:
            ax.bar(metrics, values, color='C0')
            ax.set_xticks(range(len(metrics)))
            ax.set_xticklabels(metrics, rotation=45, ha='right')
            # highlight x tick labels where value != 0
            xticks = ax.get_xticklabels()
            for lbl, nz in zip(xticks, nonzero_mask):
                lbl.set_color('black' if nz else 'lightgray')
                # if nz:
                #     lbl.set_fontweight('bold')
            ax.set_ylabel('Value')
        ax.set_title(country)

    # hide any unused axes
    for j in range(len(countries), len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle(f'Hazard: {hazard_name}', fontsize=14)
    if save:
        safe_name = ''.join(ch for ch in hazard_name if ch in (' ', '-', '_')).rstrip()
        outpath = os.path.join(plots_output_dir, f'{safe_name}.png')
        fig.savefig(outpath, dpi=150, bbox_inches='tight')
    plt.show()


## 1. Extract: Load Raw Data

Read country-level raw exports, parse XML content (or reuse the pre-extracted combined CSV), and standardize extraction settings.

In [ ]:
# Loop through ZIP files
for file in os.listdir(folder_path):
    if not file.lower().endswith(".zip"):
        continue

    zip_path = os.path.join(folder_path, file)
    base = os.path.splitext(file)[0]
    country_code = base.split("_")[-1]

    print(f"\nProcessing ZIP: {file}")

    with zipfile.ZipFile(zip_path, 'r') as z:
        # Find all XML files inside the ZIP
        xml_files = [f for f in z.namelist() if f.lower().endswith(".xml")]

        if len(xml_files) == 0:
            print("No XML found.")
            zips_without_xml.append(file)
            continue

        xml_file = xml_files[0]
        print(f"Reading XML file: {xml_file}")

        # Read XML using pandas.read_xml()
        with z.open(xml_file) as xml_stream:
            try:
                df = pd.read_xml(xml_stream, xpath=f".//{tag_name}")
            except Exception as e:
                print(f"ERROR reading XML with pandas: {e}")
                print("Skipping this ZIP.")
                zips_without_xml.append(file)
                continue
        
        # Fix text encoding issues
        df = df.applymap(fix_text)

        # Add country code column
        df['country_code'] = country_code

        # Save CSV named after ZIP
        csv_path = os.path.join(output_csv_folder, f"{base}.csv")
        df.to_csv(csv_path, index=False)
        print(f"CSV saved: '{csv_path}'")

        # Add to combined list
        all_dfs.append(df)

# Combine everything
if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)
else:
    print("No XML data processed.")

# Save CSV named after ZIP
combined_csv_path = os.path.join(output_csv_folder, "desinventar_combined.csv")
combined_df.to_csv(combined_csv_path, index=False)
print(f"CSV saved: '{combined_csv_path}'")

# Report missing XMLs
print("\n### ZIPs WITHOUT XML ###")
for missing in zips_without_xml:
    print(" -", missing)


In [ ]:

# Load saved file and review head

combined_csv_path = os.path.join(output_csv_folder, "desinventar_combined.csv")
combined_df = pd.read_csv(combined_csv_path)
combined_df.head()

In [ ]:
### CALL DEEPL API TO TRANSLATE UNIQUE EVENT TYPES ###

deepl_api_key = ""  # replace with your actual API key https://support.deepl.com/hc/en-us/articles/360020695820-API-key-for-DeepL-API
unique_event_types = combined_df["evento"].unique().tolist()
unique_event_types = [x for x in unique_event_types if isinstance(x, str)]
translator = deepl.DeepLClient(deepl_api_key)

unique_event_types_translated = []
counter = 0
for x in unique_event_types:
    x_translated = translator.translate_text(x, target_lang="EN-GB").text
    unique_event_types_translated.append(x_translated)
    counter += 1
    if counter % 10 == 0:
        time.sleep(1)  # to avoid rate limiting

# save translation mapping to csv
key = unique_event_types
value = unique_event_types_translated
event_type_translation_dict = dict(zip(key, value))
# event_type_translation_dict

df_unique_event_types_translated = pd.DataFrame.from_dict(event_type_translation_dict, orient='index', columns=['event_type_translated']).reset_index().rename(columns={'index': 'event_type_original'})
df_unique_event_types_translated.to_csv('unique_event_types_translated.csv', encoding='utf-8-sig', index=False)

## 2. Explore Data

Inspect grouped hazard mappings and confirm that reviewed event types are correctly joined before transformation.

In [ ]:
# Filter dataframe based on reviewed event types
# The reference csv file unique_event_types_reviewed is created by data analyst.
# It contains the reviewed and grouped event types that are relevant for the project, with columns 'event_type_original' and 'event_type_grouped'
reviewed_csv_path = os.path.join("unique_event_types_reviewed.csv")
reviewed_df = pd.read_csv(reviewed_csv_path)
reviewed_df.head()

combined_df = combined_df.merge(reviewed_df, left_on='evento', right_on='event_type_original', how='inner')
combined_df.dropna(subset=['event_type_grouped'], inplace=True)
combined_df

## 3. Transform: Clean and Process Data

Translate and harmonize field names, keep relevant grouped hazards, and persist the cleaned analysis dataset.

In [ ]:
# Rename columns to English
combined_df.columns
columns_original = [
    'evento', 'lugar', 'fechano', 'fechames', 'fechadia', 'muertos',
    'heridos', 'desaparece', 'afectados', 'vivdest', 'vivafec', 'otros',
    'fuentes', 'valorloc', 'valorus', 'fechapor', 'fechafec', 'hay_muertos',
    'hay_heridos', 'hay_deasparece', 'hay_afectados', 'hay_vivdest',
    'hay_vivafec', 'hay_otros', 'socorro', 'salud', 'educacion',
    'agropecuario', 'industrias', 'acueducto', 'alcantarillado', 'energia',
    'comunicaciones', 'causa', 'descausa', 'transporte', 'magnitud2',
    'nhospitales', 'nescuelas', 'nhectareas', 'cabezas', 'kmvias',
    'duracion', 'damnificados', 'evacuados', 'hay_damnificados',
    'hay_evacuados', 'hay_reubicados', 'reubicados',
]
columns_translated = [
    'event_type_original', 'location', 'date_year', 'date_month', 'date_day', 'deaths',
    'injured', 'missing', 'affected', 'homes_destroyed', 'homes_damaged', 'other_impacts',
    'sources', 'local_value', 'usd_value',  'date_reported', 'date_damaged', 'has_deaths',
    'has_injured', 'has_missing', 'has_affected', 'has_homes_destroyed',
    'has_homes_damaged', 'has_other_impacts', 'relief', 'health', 'education',
    'agriculture_livestock', 'industries', 'water_supply', 'sanitation',  'energy',
    'communications',  'cause',  'cause_description',  'transportation',  'magnitude2',
    'num_hospitals_affected',  'num_schools_affected',  'num_hectares_affected',
    'num_livestock_affected',  'km_roads_affected',  'duration_days',  'displaced_persons',
    'evacuated_persons',  'has_displaced_persons',
    'has_evacuated_persons',  'has_relocated_persons',  'relocated_persons', 
]
rename_dict = dict(zip(columns_original, columns_translated))
combined_df.rename(columns=rename_dict, inplace=True)

# Save cleaned and merged CSV
combined_df.to_csv(os.path.join(output_csv_folder, "desinventar_combined_translated_grouped.csv"), index=False)

## 6. Load: Save Results

The cleaned merged dataset is saved to `extracted/desinventar_combined_translated_grouped.csv`. Plot outputs are saved under `plots/` when `save_figures = True`.

In [ ]:
# Load saved file and review head
combined_csv_path = os.path.join(output_csv_folder, "desinventar_combined_translated_grouped.csv")
combined_df = pd.read_csv(combined_csv_path)
combined_df.head()

In [ ]:
columns_indicators = [
    'deaths',
    'affected',
    'injured',
    'missing',
    'homes_destroyed',
    'homes_damaged',
    # 'local_value',
    # 'usd_value',
    'relief',
    'health',
    'education',
    'agriculture_livestock',
    'industries',
    'water_supply',
    'sanitation',
    'energy',
    'communications',
    'transportation',
    'num_hospitals_affected',
    'num_schools_affected',
    'num_hectares_affected',
    'num_livestock_affected',
    'km_roads_affected',
    'duration_days',
    'displaced_persons',
    'evacuated_persons',
    'relocated_persons',
]


In [ ]:
# indicator value threshold - adjust when reevaluating
min_indicator_value = 0

# Convert selected columns to numeric (non-convertible -> NaN)
num_df = combined_df[columns_indicators].apply(pd.to_numeric, errors='coerce')

# Boolean DataFrame where True == value >= min_indicator_value
zeros_bool = num_df.ge(min_indicator_value)

# Group keys and group objects
group_keys = ['country_code', 'event_type_grouped']
grouped = combined_df.groupby(group_keys)

# Total rows per group (includes rows with NaN indicator values)
total_rows = grouped.size().rename('total_rows')

# Sum zeros per group (counts of rows with value == 0 per indicator)
zeros_per_group = zeros_bool.groupby([combined_df[k] for k in group_keys]).sum()

# Non-missing counts per group (per indicator)
nonmissing_per_group = num_df.groupby([combined_df[k] for k in group_keys]).count()

# Percent zeros relative to total rows and relative to non-missing values
zeros_pct_total = zeros_per_group.div(total_rows, axis=0) * 100
nonmissing_safe = nonmissing_per_group.replace(0, np.nan)
zeros_pct_nonmissing = zeros_per_group.div(nonmissing_safe) * 100

# Combine results with clear suffixes, round percentages, and flatten index
result = (
    zeros_per_group.add_suffix('_zeros')
    .join(zeros_pct_total.add_suffix('_pct_total'))
    .join(zeros_pct_nonmissing.add_suffix('_pct_nonmissing'))
)
for c in result.columns:
    if c.endswith('_pct_total') or c.endswith('_pct_nonmissing'):
        result[c] = result[c].round(2)

result = result.reset_index().sort_values(group_keys).reset_index(drop=True)

# Show result and (optionally) save
display(result)
# result.to_csv('zeros_per_country_eventtype.csv', index=False)  # uncomment to save

## 3.1 Visual Check (Optional)

Visualize completeness-based indicator diagnostics (non-missing percentages) by hazard and country.

In [ ]:
# ### PLOT INDICATORS PER HAZARD (OPTIONAL) ###

# id_cols = ['country_code', 'event_type_grouped']
# exclude_cols = ['country_code', 'event_type_grouped', 'uu_id'] + [c for c in result.columns if c.endswith('_zeros')] + [c for c in result.columns if c.endswith('_pct_total')]
# plot_cols = [c for c in result.columns if c not in exclude_cols and c not in id_cols]
# print('plot_cols: ', plot_cols)
# melted = result.melt(id_vars=id_cols, value_vars=plot_cols, var_name='indicator', value_name='value')
# melted = melted.dropna(subset=['value'])
# melted = melted.round(2)
# print('melted: ', melted)

# # Configuration: change these values as needed
# plots_output_dir = 'plots'
# os.makedirs(plots_output_dir, exist_ok=True)
# plot_ncols = 4
# figsize_per_subplot = (4, 2.5)  # (width, height) per subplot
# orientation = 'vertical'  # 'vertical' or 'horizontal'
# normalize = None  # None, 'metric', or 'country'
# min_metric_value = 0.0  # filter out metrics whose max absolute value <= this
# save_figures = True

# def plot_indicator(hazard_name, df_melted, ncols=plot_ncols, figsize_per=(5,3), orientation=orientation, normalize=normalize, min_metric_value=min_metric_value, save=save_figures):
#     df_h = df_melted[df_melted['event_type_grouped'] == hazard_name].copy()

#     # pivot to indicator x country
#     pivot = df_h.pivot_table(index='indicator', columns='country_code', values='value', aggfunc='first').fillna(0)
#     countries = [c for c in pivot.columns.tolist() if pivot[c].abs().max() > min_metric_value]
#     if len(countries) == 0:
#         print(f'No countries above threshold for hazard: {hazard_name}')
#         return
#     pivot = pivot[countries]

#     # normalization options
#     metrics = pivot.index.tolist()
#     if normalize == 'metric':
#         for m in metrics:
#             mx = pivot[m].abs().max()
#             if mx > 0:
#                 pivot[m] = pivot[m] / mx
#     elif normalize == 'country':
#         pivot = pivot.div(pivot.max(axis=1).replace(0, 1), axis=0)

#     countries = pivot.index.tolist()
#     n_countries = len(countries)
#     ncols = max(1, ncols)
#     nrows = math.ceil(n_countries / ncols)

#     fig_w = ncols * figsize_per[0]
#     fig_h = nrows * figsize_per[1]
#     fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), squeeze=False, constrained_layout=True)
#     axes_flat = axes.flatten()

#     for i, country in enumerate(countries):
#         ax = axes_flat[i]
#         values = pivot.loc[country, metrics].values
#         nonzero_mask = (np.abs(values) > 0)  # mask of metrics that have non-zero values for this country"
#         if orientation == 'horizontal':
#             ax.barh(metrics, values, color='C0')
#             # highlight y tick labels where value != 0"
#             yticks = ax.get_yticklabels()
#             for lbl, nz in zip(yticks, nonzero_mask):
#                 lbl.set_color('black' if nz else 'lightgray')
#                 # if nz:
#                 #     lbl.set_fontweight('bold')
#         else:
#             ax.bar(metrics, values, color='C0')
#             ax.set_xticks(range(len(metrics)))
#             ax.set_xticklabels(metrics, rotation=45, ha='right')
#             # highlight x tick labels where value != 0
#             xticks = ax.get_xticklabels()
#             for lbl, nz in zip(xticks, nonzero_mask):
#                 lbl.set_color('black' if nz else 'lightgray')
#                 # if nz:
#                 #     lbl.set_fontweight('bold')
#             ax.set_ylabel('Value')
#         ax.set_title(country)

#     # hide any unused axes
#     for j in range(len(countries), len(axes_flat)):
#         axes_flat[j].set_visible(False)

#     fig.suptitle(f'Hazard: {hazard_name}', fontsize=14)
#     if save:
#         safe_name = ''.join(ch for ch in hazard_name if ch in (' ', '-', '_')).rstrip()
#         outpath = os.path.join(plots_output_dir, f'{safe_name}.png')
#         fig.savefig(outpath, dpi=150, bbox_inches='tight')
#     plt.show()

# # Plot all hazards and save figures
# hazards = melted['event_type_grouped'].unique()
# for hazard in hazards:
#     plot_indicator(
#         hazard,
#         melted,
#         ncols=plot_ncols,
#         figsize_per=figsize_per_subplot,
#         orientation=orientation,
#         normalize=normalize,
#         min_metric_value=min_metric_value,
#         save=save_figures)

## 4. Aggregate Data

Aggregate indicator values by country and grouped hazard type using event-level summaries.

In [ ]:
aggregation ={'uu_id': 'count',
     'deaths': 'mean',
     'affected': 'mean',
     'injured': 'mean',
     'missing': 'mean',
     'affected': 'mean',
     'homes_destroyed': 'mean', 
     'homes_damaged': 'mean', 
    # 'local_value': 'mean', 
    # 'usd_value': 'mean', 
    'relief': 'mean', 
    'health': 'mean', 
    'education': 'mean',
    'agriculture_livestock': 'mean', 'industries': 'mean', 'water_supply': 'mean', 'sanitation': 'mean',  'energy': 'mean',
    'communications': 'mean',  'transportation': 'mean', 
    'num_hospitals_affected': 'mean',  'num_schools_affected': 'mean',  'num_hectares_affected': 'mean',
    'num_livestock_affected': 'mean',  'km_roads_affected': 'mean',  'duration_days': 'mean',  'displaced_persons': 'mean',
    'evacuated_persons': 'mean',  'relocated_persons': 'mean', }
summarize = combined_df.groupby(['country_code', 'event_type_grouped']).agg(aggregation).reset_index()


## 5. Visualize (Optional)

Generate hazard-specific comparison plots across countries from aggregated indicator values.

In [ ]:
### PLOT INDICATORS/COUNTRY PER HAZARD ###

id_cols = ['country_code', 'event_type_grouped']
exclude_cols = ['country_code', 'event_type_grouped', 'uu_id']
plot_cols = [c for c in summarize.columns if c not in exclude_cols and c not in id_cols]
# ensure numeric where possible
for c in plot_cols:
    summarize[c] = pd.to_numeric(summarize[c], errors='coerce')
melted = summarize.melt(id_vars=id_cols, value_vars=plot_cols, var_name='metric', value_name='value')
melted = melted.dropna(subset=['value'])
melted = melted.round(2)
melted


# Configuration: change these values as needed
plot_ncols = 4
figsize_per_subplot = (4, 2.5)  # (width, height) per subplot
orientation = 'vertical'  # 'vertical' or 'horizontal'
normalize = None  # None, 'metric', or 'country'
min_metric_value = 0.0  # filter out metrics whose max absolute value <= this
save_figures = True

# Plot all hazards and save figures
hazards = melted['event_type_grouped'].unique()
for hazard in hazards:
    plot_hazard(
        hazard,
        melted,
        ncols=plot_ncols,
        figsize_per=figsize_per_subplot,
        orientation=orientation,
        normalize=normalize,
        min_metric_value=min_metric_value,
        save=save_figures)


## Summary

This notebook:

- loaded and harmonized DesInventar event records from multiple countries
- mapped raw event labels to reviewed hazard groups
- computed completeness and average-impact indicators by country and hazard
- generated hazard-level comparison plots and saved outputs

**Next steps**: validate indicator thresholds and decide which metrics to prioritize for risk profiling.